In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "PAQ_C.xpt"
csv_file = DATA_DIR / "PAQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched copy for final fidelity check
source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 PAQ_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nVariables:")
print(df.columns.tolist())


# ============================================================
# 3. EXPECTED RELEASED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PAD020",
    "PAQ050Q",
    "PAQ050U",
    "PAD080",
    "PAQ100",
    "PAD120",
    "PAD160",
    "PAQ180",
    "PAD200",
    "PAD320",
    "PAD440",
    "PAD460",
    "PAQ500",
    "PAQ520",
    "PAQ540",
    "PAQ560",
    "PAD590",
    "PAD600",
    "PAAQUEX"
]


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "PAQ_C column names/order differ from expected."
    )

print(
    "\nPASS: Column names/order match PAQ_C."
)


# ============================================================
# 4. STRUCTURE CHECK
# ============================================================

if len(df) != 9278:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )

if len(df.columns) != 20:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )

print(
    "PASS: Structure = 9,278 rows x 20 columns."
)


# ============================================================
# 5. KNOWN PANDAS/XPORT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. CDC-VALIDATED ZERO VARIABLES
# ============================================================

#
# PAQ560:
#     CDC says IF NEVER, ENTER 0.
#
# PAD590:
#     0 = Less than 1 hour
#     CDC count = 1,122
#
# PAD600:
#     0 = Less than 1 hour
#     CDC count = 2,481
#

ZERO_ALLOWED_COLUMNS = [
    "PAQ560",
    "PAD590",
    "PAD600"
]


EXPECTED_TINY_COUNTS = {
    "PAQ560": 111,
    "PAD590": 1122,
    "PAD600": 2481
}


# ============================================================
# 7. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)

approved_tiny_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected = (
    actual_tiny_columns
    -
    approved_tiny_columns
)


if unexpected:

    raise ValueError(
        "STOP: Tiny values found in unexpected PAQ fields:\n"
        + str(sorted(unexpected))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts.get(
            col,
            0
        )
    )

    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count} tiny values, "
            f"found {actual_count}."
        )


print(
    "\nPASS: All 3,714 tiny values occur only "
    "in CDC zero-permitted variables."
)


# ============================================================
# 8. RESTORE TRUE ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col in ZERO_ALLOWED_COLUMNS:

    mask = df[col].eq(
        TINY_VALUE
    )

    count = int(
        mask.sum()
    )

    print(
        f"{col}: {count:,} tiny values -> 0"
    )

    df.loc[
        mask,
        col
    ] = 0.0


# ============================================================
# 9. VERIFY ZERO COUNTS
# ============================================================

print("\n--- ZERO COUNTS AFTER RESTORATION ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_zero = int(
        df[col]
        .eq(0)
        .sum()
    )

    print(
        f"{col}: {actual_zero:,}"
    )

    if actual_zero != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count} zeros, "
            f"found {actual_zero}."
        )


print(
    "PASS: 3,714 zeros restored correctly."
)


# ============================================================
# 10. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# IMPORTANT:
#
# Do NOT round any genuine decimal.
#
# First check every numeric source field.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nReal decimal values found in {col}:"
        )

        print(
            values[
                fractional_mask
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values exist. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values exist "
    "in this PAQ_C XPT."
)


# ============================================================
# 11. CLEAN SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 12. FORMAT PAQ/PAD FIELDS WITHOUT .0
# ============================================================

#
# Every PAQ/PAD value was checked above and
# all values are whole-number values.
#
# No .round() is used.
#

PAQ_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in PAQ_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: PAQ/PAD fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 13. CDC CATEGORICAL FREQUENCY CHECKS
# ============================================================

EXPECTED_CATEGORICAL = {

    "PAD020": {
        "counts": {
            1: 2318,
            2: 4716,
            3: 207,
            7: 1
        },
        "missing": 2036
    },

    "PAQ050U": {
        "counts": {
            1: 627,
            2: 1018,
            3: 673
        },
        "missing": 6960
    },

    "PAQ100": {
        "counts": {
            1: 3455,
            2: 2561,
            3: 194,
            7: 1,
            9: 2
        },
        "missing": 3065
    },

    "PAQ180": {
        "counts": {
            1: 1554,
            2: 3188,
            3: 1030,
            4: 436,
            7: 1,
            9: 4
        },
        "missing": 3065
    },

    "PAD200": {
        "counts": {
            1: 2759,
            2: 4158,
            3: 324,
            7: 1
        },
        "missing": 2036
    },

    "PAD320": {
        "counts": {
            1: 4035,
            2: 2967,
            3: 236,
            7: 1,
            9: 1
        },
        "missing": 2038
    },

    "PAD440": {
        "counts": {
            1: 2569,
            2: 4458,
            3: 211,
            7: 1
        },
        "missing": 2039
    },

    "PAQ500": {
        "counts": {
            1: 1499,
            2: 1569,
            3: 4164,
            7: 1,
            9: 5
        },
        "missing": 2040
    },

    "PAQ520": {
        "counts": {
            1: 2532,
            2: 1351,
            3: 3279,
            7: 2,
            9: 74
        },
        "missing": 2040
    },

    "PAQ540": {
        "counts": {
            1: 485,
            2: 2675,
            3: 966,
            7: 1,
            9: 3
        },
        "missing": 5148
    },

    "PAD590": {
        "counts": {
            0: 1122,
            1: 1528,
            2: 2478,
            3: 1593,
            4: 901,
            5: 1441,
            6: 131,
            77: 1,
            99: 2
        },
        "missing": 81
    },

    "PAD600": {
        "counts": {
            0: 2481,
            1: 1339,
            2: 728,
            3: 337,
            4: 146,
            5: 225,
            6: 3936,
            77: 1,
            99: 4
        },
        "missing": 81
    },

    "PAAQUEX": {
        "counts": {
            1: 8173,
            2: 1105
        },
        "missing": 0
    }
}


print("\n--- CDC CATEGORICAL FREQUENCY CHECK ---")


for col, spec in EXPECTED_CATEGORICAL.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if (
        actual_counts != spec["counts"]
        or
        actual_missing != spec["missing"]
    ):

        print(
            f"\nFAIL: {col}"
        )

        print(
            "Expected:",
            spec["counts"]
        )

        print(
            "Actual:",
            actual_counts
        )

        print(
            "Expected missing:",
            spec["missing"]
        )

        print(
            "Actual missing:",
            actual_missing
        )

        raise ValueError(
            f"{col} differs from CDC."
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 14. RANGE CHECK HELPER
# ============================================================

def check_range(
    col,
    range_min,
    range_max,
    range_count,
    missing_count,
    special_counts=None
):

    if special_counts is None:
        special_counts = {}

    s = df[col]


    actual_range_count = int(
        s.between(
            range_min,
            range_max
        ).sum()
    )


    actual_missing = int(
        s.isna()
        .sum()
    )


    print(f"\n--- {col} ---")

    print(
        f"{range_min}-{range_max}:",
        actual_range_count
    )

    print(
        "Missing:",
        actual_missing
    )


    if actual_range_count != range_count:

        raise ValueError(
            f"{col}: range count differs."
        )


    if actual_missing != missing_count:

        raise ValueError(
            f"{col}: missing count differs."
        )


    for code, expected_count in special_counts.items():

        actual_count = int(
            s.eq(code)
            .sum()
        )

        print(
            f"{code}:",
            actual_count
        )


        if actual_count != expected_count:

            raise ValueError(
                f"{col}: code {code} count differs."
            )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 15. PAQ050Q
# ============================================================

check_range(
    col="PAQ050Q",
    range_min=1,
    range_max=124,
    range_count=2318,
    missing_count=6960,
    special_counts={
        77777: 0,
        99999: 0
    }
)


# ============================================================
# 16. PAD080
# ============================================================

check_range(
    col="PAD080",
    range_min=1,
    range_max=600,
    range_count=2310,
    missing_count=6968,
    special_counts={
        77777: 0,
        99999: 0
    }
)


# ============================================================
# 17. PAD120
# ============================================================

check_range(
    col="PAD120",
    range_min=1,
    range_max=300,
    range_count=3447,
    missing_count=5828,
    special_counts={
        77777: 0,
        99999: 3
    }
)


# ============================================================
# 18. PAD160
# ============================================================

check_range(
    col="PAD160",
    range_min=1,
    range_max=600,
    range_count=3443,
    missing_count=5835,
    special_counts={
        77777: 0,
        99999: 0
    }
)


# ============================================================
# 19. PAD460
# ============================================================

check_range(
    col="PAD460",
    range_min=1,
    range_max=300,
    range_count=2567,
    missing_count=6709,
    special_counts={
        777: 0,
        999: 2
    }
)


# ============================================================
# 20. PAQ560
# ============================================================

#
# CDC:
#
# 0-77    = 1,953
# 77777   = 0
# 99999   = 6
# missing = 7,319
#
# In THIS source XPT, 111 of those values are zero.
#

check_range(
    col="PAQ560",
    range_min=0,
    range_max=77,
    range_count=1953,
    missing_count=7319,
    special_counts={
        77777: 0,
        99999: 6
    }
)


if int(df["PAQ560"].eq(0).sum()) != 111:

    raise ValueError(
        "PAQ560 should contain 111 restored zeros "
        "in this XPT."
    )


print(
    "PASS: PAQ560 zero count = 111."
)


# ============================================================
# 21. BUILD EXPECTED SOURCE
# ============================================================

#
# Expected source = original XPT with ONLY:
#
# PAQ560 tiny -> 0
# PAD590 tiny -> 0
# PAD600 tiny -> 0
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 22. VERIFY EVERY VALUE BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


value_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(value_failures)
    )


print(
    "PASS: All 20 variables preserved exactly "
    "after validated zero restoration."
)


# ============================================================
# 23. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Remaining tiny values:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifacts remain."
    )


print(
    "PASS: No tiny zero artifact remains."
)


# ============================================================
# 24. EXPORT CSV
# ============================================================

#
# No float_format.
# No recalculation.
# No arbitrary rounding.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 25. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 26. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 20 columns match expected XPT values."
)


# ============================================================
# 27. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 28. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


print("\n" + "=" * 75)
print("PAQ_C CONVERSION COMPLETE")
print("=" * 75)

NHANES 2003-2004 PAQ_C XPT -> CSV
Rows: 9,278
Columns: 20

Variables:
['SEQN', 'PAD020', 'PAQ050Q', 'PAQ050U', 'PAD080', 'PAQ100', 'PAD120', 'PAD160', 'PAQ180', 'PAD200', 'PAD320', 'PAD440', 'PAD460', 'PAQ500', 'PAQ520', 'PAQ540', 'PAQ560', 'PAD590', 'PAD600', 'PAAQUEX']

PASS: Column names/order match PAQ_C.
PASS: Structure = 9,278 rows x 20 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 3,714

Tiny values by variable:
PAD600    2481
PAD590    1122
PAQ560     111

PASS: All 3,714 tiny values occur only in CDC zero-permitted variables.

--- ZERO RESTORATION ---
PAQ560: 111 tiny values -> 0
PAD590: 1,122 tiny values -> 0
PAD600: 2,481 tiny values -> 0

--- ZERO COUNTS AFTER RESTORATION ---
PAQ560: 111
PAD590: 1,122
PAD600: 2,481
PASS: 3,714 zeros restored correctly.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in this PAQ_C XPT.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31125

PASS: PAQ/PAD fiel